# 03 — Model optimisation & training (Step 3)
Train and tune **two models** — a **Neural Network (Keras)** and a **Random Forest (scikit-learn)** — for
each scenario: the **X=0 no-neighbour baseline** plus every `X` × neighbour-encoding (`pos` / `agg`)
combination. Hyperparameters are tuned with **cross-validation** (kept light to respect the ~30-min
Colab budget). We record MSE/MAE/R² and **training duration**.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); keep >= 10
                                 # so the nearest-alignment tolerance stays above the worst raw gap.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): at full population the top ~1% of active samples carry
                                 # ~86% of the total variance -> MSE/R2 would be about a handful of rare peaks.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    return {"MSE": float(mean_squared_error(y_true, y_pred)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2":  float(r2_score(y_true, y_pred))}

In [ ]:
import json, time, joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
tf.random.set_seed(RANDOM_SEED)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

def stem(x, enc):
    """File stem used by notebook 02: acc_X0 (baseline), acc_X{x} (pos), acc_X{x}_agg (agg)."""
    return f"acc_X{x}" + ("_agg" if enc == "agg" else "")

def load_xy(x, enc="pos"):
    d = np.load(f"{PROCESSED_DIR}/{stem(x, enc)}.npz")
    return d["X_train"], d["y_train"], d["X_val"], d["y_val"], d["X_test"], d["y_test"]

## Neural network
Small MLP. We do a tiny grid over width/learning-rate using the validation set, then keep the best.

In [ ]:
def build_mlp(input_dim, units=64, lr=1e-3, depth=2):
    m = keras.Sequential([keras.layers.Input((input_dim,))])
    for _ in range(depth):
        m.add(keras.layers.Dense(units, activation="relu"))
    m.add(keras.layers.Dense(1))
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="mse", metrics=["mae"])
    return m

def train_nn(Xtr, ytr, Xva, yva):
    best, best_val, best_cfg, best_hist = None, np.inf, None, None
    es = keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)
    for units in (64, 128):                       # small validation-based search (GPU: a few seconds each)
        m = build_mlp(Xtr.shape[1], units=units, lr=1e-3)
        h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=30, batch_size=1024,
                  callbacks=[es], verbose=0)
        v = m.evaluate(Xva, yva, verbose=0)[0]
        if v < best_val:
            best, best_val, best_cfg, best_hist = m, v, {"units": units, "lr": 1e-3}, h.history
    return best, best_cfg, best_hist              # history -> learning-curve figure below

## Random forest
`GridSearchCV` (3-fold) over a small grid satisfies the cross-validation requirement.

**Speed (without hurting validity):** the default `max_features` makes RF regression evaluate *all* features
at every split → very slow with 73 features on Colab's 2 vCPUs. We use `max_features="sqrt"` (standard, faster,
usually better), `max_samples=0.3` (each tree fits on a 30% bootstrap subsample — this is just bagging), and
**tune on a capped subsample**, then refit the best config on the full training set. Benchmarked on the real
data (270k train rows): accuracy is essentially unchanged while fitting ~4× faster.

In [ ]:
RF_TUNE_SUBSAMPLE = 20000   # rows used for the CV search (final model is refit on ALL training rows)

def train_rf(Xtr, ytr):
    if len(Xtr) > RF_TUNE_SUBSAMPLE:
        s = np.random.RandomState(RANDOM_SEED).choice(len(Xtr), RF_TUNE_SUBSAMPLE, replace=False)
        Xt, yt = Xtr[s], ytr[s]
    else:
        Xt, yt = Xtr, ytr
    base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1,
                                 max_features="sqrt", min_samples_leaf=2, max_samples=0.3)
    grid = {"n_estimators": [100, 200], "max_depth": [16, None]}
    gs = GridSearchCV(base, grid, cv=3, scoring="neg_mean_squared_error", n_jobs=1)
    gs.fit(Xt, yt)
    best = gs.best_estimator_
    best.fit(Xtr, ytr)                      # refit best config on the full training set
    return best, gs.best_params_

## Train both models for every scenario and save results
Scenarios: `(X=0, none)` baseline + `X ∈ X_VALUES` × `enc ∈ {pos, agg}` — 7 datasets × 2 models.
The baseline tells us how much the closest-users features add at all; pos-vs-agg tells us which
representation of the neighbours works better under heavy co-location.

In [ ]:
def infer_ms(predict, X):
    t = time.time(); predict(X); return round((time.time()-t)/len(X)*1000, 4)   # ms per sample

SCENARIOS = [(0, "none")] + [(x, e) for x in X_VALUES for e in ENCODINGS]

results, nn_histories = [], {}
for x, enc in SCENARIOS:
    Xtr, ytr, Xva, yva, Xte, yte = load_xy(x, enc)
    s = stem(x, enc)

    t = time.time(); nn, nn_cfg, nn_hist = train_nn(Xtr, ytr, Xva, yva); nn_dur = time.time()-t
    nn_histories[(x, enc)] = nn_hist
    nn_pred = nn.predict(Xte, verbose=0).ravel()
    nn_m = evaluate(yte, nn_pred); nn_m.update(model="NN", X=x, enc=enc, train_s=round(nn_dur,1),
        infer_ms=infer_ms(lambda X: nn.predict(X, verbose=0), Xte), cfg=str(nn_cfg))
    nn.save(f"{RESULTS_DIR}/models/nn_{s.removeprefix('acc_')}.keras")

    t = time.time(); rf, rf_cfg = train_rf(Xtr, ytr); rf_dur = time.time()-t
    rf_pred = rf.predict(Xte)
    rf_m = evaluate(yte, rf_pred); rf_m.update(model="RF", X=x, enc=enc, train_s=round(rf_dur,1),
        infer_ms=infer_ms(rf.predict, Xte), cfg=str(rf_cfg))
    joblib.dump(rf, f"{RESULTS_DIR}/models/rf_{s.removeprefix('acc_')}.pkl")

    results += [nn_m, rf_m]
    print(f"X={x:>2} {enc:<4} | NN R2={nn_m['R2']:.3f} ({nn_dur:.0f}s) | RF R2={rf_m['R2']:.3f} ({rf_dur:.0f}s)")

import pandas as pd
res = pd.DataFrame(results)
res.to_csv(f"{RESULTS_DIR}/metrics.csv", index=False)
res

## NN learning curves (training diagnostics)
Training vs validation loss for the selected network of every scenario. What to check: validation tracks
training (no overfitting — early stopping with `restore_best_weights` picks the minimum), and curves have
flattened (the epoch budget is sufficient).

In [ ]:
import matplotlib.pyplot as plt
GREEN, RED, INK, MUT = "#2a9d8f", "#e76f51", "#333333", "#777777"   # project palette

n_sc = len(nn_histories)
ncols = (n_sc + 1) // 2
fig, axes = plt.subplots(2, ncols, figsize=(3.1 * ncols, 6), sharey=True)
axes = np.atleast_1d(axes).ravel()
for a, ((x, enc), h) in zip(axes, nn_histories.items()):
    a.plot(h["loss"], color=MUT, lw=1.5)
    a.plot(h["val_loss"], color=GREEN, lw=2)
    a.set_title(f"X={x} {enc}", fontsize=10)
    a.set_xlabel("epoch")
    a.spines[["top", "right"]].set_visible(False)
    a.grid(alpha=.25); a.set_axisbelow(True)
axes[0].set_ylabel("MSE loss")
axes[0].legend(["train", "validation"], frameon=False, labelcolor=INK, fontsize=9)
for a in axes[n_sc:]:
    a.axis("off")
fig.suptitle("NN training: validation follows training in every scenario", y=1.0)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/03_nn_learning_curves.png", dpi=150); plt.show()

## Learning curve: does more data help?
R² on the fixed test set as the training set grows (baseline scenario, both models). A flat right end means
the models are **data-saturated**: the residual error is irreducible with these features, not a data-volume
problem — which supports the demand-driven reading (the missing information is the user's intent, not more
samples). Set `RUN_LEARNING_CURVE = False` to skip (adds a few minutes).

In [ ]:
RUN_LEARNING_CURVE = True
FRACTIONS = [0.05, 0.1, 0.25, 0.5, 1.0]

if RUN_LEARNING_CURVE:
    from sklearn.ensemble import RandomForestRegressor
    Xtr, ytr, Xva, yva, Xte, yte = load_xy(0, "none")     # baseline: own features only
    rng_lc = np.random.default_rng(RANDOM_SEED)
    rows_lc = []
    for f in FRACTIONS:
        n = int(len(Xtr) * f)
        s = rng_lc.choice(len(Xtr), n, replace=False)
        rf_lc = RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1,
                                      max_features="sqrt", min_samples_leaf=2, max_samples=0.3)
        rf_lc.fit(Xtr[s], ytr[s])
        r2_rf = evaluate(yte, rf_lc.predict(Xte))["R2"]
        keras.utils.set_random_seed(RANDOM_SEED)
        nn_lc = build_mlp(Xtr.shape[1], units=64, lr=1e-3)
        nn_lc.fit(Xtr[s], ytr[s], validation_data=(Xva, yva), epochs=30, batch_size=1024,
                  callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)], verbose=0)
        r2_nn = evaluate(yte, nn_lc.predict(Xte, verbose=0).ravel())["R2"]
        rows_lc.append((n, r2_rf, r2_nn))
        print(f"n_train={n:>7d} | RF R2={r2_rf:.3f} | NN R2={r2_nn:.3f}")

    lc = pd.DataFrame(rows_lc, columns=["n_train", "RF", "NN"])
    fig, ax = plt.subplots(figsize=(7, 4.4))
    for model, color in [("NN", GREEN), ("RF", RED)]:
        ax.plot(lc.n_train, lc[model], marker="o", ms=6, lw=2, color=color)
        ax.text(lc.n_train.iloc[-1] * 1.04, lc[model].iloc[-1], model, color=INK, va="center")
    ax.set_xscale("log")
    ax.set_xlabel("training samples (log scale)"); ax.set_ylabel("R\u00b2 (test)")
    ax.set_title("Both models saturate before the full training set:\nthe residual error is not a data-volume problem")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=.25); ax.set_axisbelow(True)
    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/03_learning_curve.png", dpi=150); plt.show()

Metrics (now including the `enc` column) saved to `results/metrics.csv`; models to `results/models/`
(`*_X0*` = baseline, `*_X{x}*` = positional, `*_X{x}_agg*` = aggregated). Notebook **04** compares them.